# 03 - Analise de Features Avancadas

Analise detalhada das features avancadas: correlacoes, informacao mutua,
multicolinearidade e importancia. Usa o modelo avancado pre-treinado.

**Nota:** Requer `python scripts/retrain.py` executado previamente.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import json
import warnings

from sklearn.feature_selection import mutual_info_regression

from src.features.feature_engineering import FeatureEngineer, STANDARD_PRESSURE_HPA
from src.models.evaluation import ModelEvaluator

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print(f"Pressao atmosferica padrao: {STANDARD_PRESSURE_HPA} hPa")

## 1. Carregar Dados e Modelo Avancado

In [ ]:
df = pd.read_parquet('../data/processed/processed_data.parquet')
print(f"Dados: {len(df):,} linhas")

fe = FeatureEngineer()
df_features = fe.create_all_features(df, use_advanced=True)
print(f"Apos FE avancado: {df_features.shape[0]:,} linhas, {df_features.shape[1]} colunas")

# Carregar modelo avancado pre-treinado
models_dir = Path('../data/models')
model_path = models_dir / 'checkpoints' / 'best_model_advanced.pkl'

if model_path.exists():
    model = joblib.load(model_path)
    with open(models_dir / 'features' / 'advanced_feature_names.txt') as f:
        saved_features = [l.strip() for l in f.readlines() if l.strip()]
    with open(models_dir / 'metadata' / 'metadata_advanced.json') as f:
        meta = json.load(f)
    print(f"Modelo avancado carregado: {meta.get('best_model', 'N/A')}")
    print(f"Features do modelo: {len(saved_features)}")
else:
    print("AVISO: Modelo avancado nao encontrado. Executar 'python scripts/retrain.py' primeiro.")
    saved_features = []
    model = None

# Listar todas as features numericas
exclude_cols = ['timestamp', 'consumption_mw', 'region', 'holiday_name', 'city', 'date']
for col in df_features.columns:
    if col in exclude_cols:
        continue
    if pd.api.types.is_datetime64_any_dtype(df_features[col]) or df_features[col].dtype == 'object':
        exclude_cols.append(col)

all_features = [c for c in df_features.columns if c not in exclude_cols]
print(f"Total features numericas: {len(all_features)}")

## 2. Analise de Features Derivadas

In [ ]:
weather_derived = ['heat_index', 'dew_point', 'comfort_index', 'effective_temperature',
                   'temp_humidity_ratio', 'wind_chill', 'pressure_relative', 'solar_proxy',
                   'precip_temp_index']

available_wd = [f for f in weather_derived if f in df_features.columns]
print("Features meteorologicas derivadas:")
for f in available_wd:
    corr = df_features[f].corr(df_features['consumption_mw'])
    print(f"  {f:30s} corr={corr:+.4f}  mean={df_features[f].mean():.2f}  std={df_features[f].std():.2f}")

trend_features = [c for c in all_features if any(x in c for x in ['diff', 'momentum', 'deviation', 'volatility'])]
print(f"\nFeatures de tendencia: {len(trend_features)}")
for f in trend_features:
    corr = df_features[f].corr(df_features['consumption_mw'])
    print(f"  {f:30s} corr={corr:+.4f}")

## 3. Correlacao com Target

In [ ]:
correlations = df_features[all_features].corrwith(df_features['consumption_mw']).abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
top30_corr = correlations.head(30)
ax.barh(range(len(top30_corr)), top30_corr.values, color='steelblue', edgecolor='black')
ax.set_yticks(range(len(top30_corr)))
ax.set_yticklabels(top30_corr.index)
ax.invert_yaxis()
ax.set_xlabel('|Correlacao| com consumo')
ax.set_title('Top 30 Features por Correlacao')
plt.tight_layout()
plt.show()

print("Top 10 features mais correlacionadas:")
for feat, corr in correlations.head(10).items():
    print(f"  {feat:40s} |r| = {corr:.4f}")

## 4. Informacao Mutua

In [ ]:
sample_size = min(50000, len(df_features))
df_sample = df_features.sample(sample_size, random_state=42)

X_sample = df_sample[all_features].fillna(0).values
y_sample = df_sample['consumption_mw'].values

mi_scores = mutual_info_regression(X_sample, y_sample, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({'feature': all_features, 'mi_score': mi_scores}).sort_values('mi_score', ascending=False)

print("Top 20 features por Informacao Mutua:")
for _, row in mi_df.head(20).iterrows():
    print(f"  {row['feature']:40s} MI = {row['mi_score']:.4f}")

fig, ax = plt.subplots(figsize=(12, 8))
top20_mi = mi_df.head(20)
ax.barh(range(len(top20_mi)), top20_mi['mi_score'].values, color='coral', edgecolor='black')
ax.set_yticks(range(len(top20_mi)))
ax.set_yticklabels(top20_mi['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Mutual Information')
ax.set_title('Top 20 Features por MI')
plt.tight_layout()
plt.show()

## 5. Multicolinearidade

In [ ]:
non_lag_features = [f for f in all_features if 'lag' not in f and 'rolling' not in f]
corr_matrix = df_features[non_lag_features].corr()

high_corr_pairs = []
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        if abs(corr_matrix.iloc[i, j]) > 0.95:
            high_corr_pairs.append((corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

print(f"Pares com |correlacao| > 0.95: {len(high_corr_pairs)}")
for f1, f2, corr in sorted(high_corr_pairs, key=lambda x: -abs(x[2]))[:15]:
    print(f"  {f1:35s} <-> {f2:35s} r={corr:.4f}")

## 6. Feature Importance do Modelo Pre-Treinado

In [ ]:
if model is not None and hasattr(model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': saved_features,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    fig, ax = plt.subplots(figsize=(12, 8))
    top20 = importance_df.head(20)
    ax.barh(range(len(top20)), top20['importance'].values, color='green', edgecolor='black')
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20['feature'].values)
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance')
    ax.set_title('Top 20 Features por Importancia (Modelo Avancado)')
    plt.tight_layout()
    plt.show()

    print("\nTop 15 features do modelo avancado:")
    for _, row in importance_df.head(15).iterrows():
        print(f"  {row['feature']:40s} importance={row['importance']:.4f}")
else:
    print("Modelo avancado nao disponivel para analise de importancia.")